# Shapes: loading 3D geometry files

The `shape` node renders a mesh from a 3D geometry resource, in **VTP** (VTK PolyData),
**PLY** or **OBJ** format.

The node itself has no parameters: color is applied with a child `color` node, and
`opacity`, `clip`, `transform`, `instance` and `focus` attach to it as well. Options that only
make sense for one format travel as *custom properties*, prefixed by that format.


In [ ]:
import { createBuilder, molstarNotebook } from "../../molviewspec-ts/mod.ts";

## A mesh with a uniform color

`download` -> `parse` -> `shape`, exactly like `volume`. The format comes from the `parse`
node, so it is never repeated on the shape itself.

In [ ]:
const builder1 = createBuilder();

builder1
  .download({ url: "https://raw.githubusercontent.com/molstar/mol-view-spec/master/test-data/shapes/capsid_tiling.vtp" })
  .parse({ format: "vtp" })
  .shape()
  .color({ color: "#3b82f6" });

molstarNotebook(builder1.getState());

## Coloring a VTP by one of its data arrays

A VTP can carry per-point and per-cell data arrays. `vtp_attribute` selects one,
`vtp_attribute_source` says whether it is a point or a cell array, and `vtp_palette` names the
color list to map the values through. A per-cell value is assigned to each vertex as the
arithmetic mean over all triangles incident to that vertex.

Custom properties are the second argument to `.shape()`.

In [ ]:
const builder2 = createBuilder();

builder2
  .download({ url: "https://raw.githubusercontent.com/molstar/mol-view-spec/master/test-data/shapes/capsid_tiling.vtp" })
  .parse({ format: "vtp" })
  .shape({}, {
    vtp_attribute: "tile_id",
    vtp_attribute_source: "cell",
    vtp_palette: "turbo",
  });

molstarNotebook(builder2.getState());

### Pinning the color range

By default the scale spans the minimum and maximum of the values, so two scenes built from
different files are not directly comparable. `vtp_domain` fixes it explicitly.

In [ ]:
const builder3 = createBuilder();

builder3
  .download({ url: "https://raw.githubusercontent.com/molstar/mol-view-spec/master/test-data/shapes/capsid_tiling.vtp" })
  .parse({ format: "vtp" })
  .shape({}, {
    vtp_attribute: "tile_id",
    vtp_attribute_source: "cell",
    vtp_palette: "viridis",
    vtp_domain: [0, 100],
  })
  .opacity(0.9);

molstarNotebook(builder3.getState());

## PLY and OBJ

Both formats can carry their own colors. `ply_coloring` and `obj_coloring` select whether to
use them; the default is `uniform`, i.e. the `color` child.

For OBJ, `custom` colors each material group by name. The names come from the OBJ's own
`usemtl` directives, so no MTL file is needed.

```typescript
// PLY, using the per-vertex colors stored in the file
builder.download({ url: "scan.ply" }).parse({ format: "ply" })
  .shape({}, { ply_coloring: "vertex" });

// OBJ, one color per material group
builder.download({ url: "cell.obj" }).parse({ format: "obj" })
  .shape({}, {
    obj_coloring: "custom",
    obj_material_colors: { membrane: "#ff3b30", cytosol: "steelblue" },
  });
```
